## Imports

In [50]:
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from sklearn.cluster import DBSCAN

from itertools import product
import matplotlib.pyplot as plt
from matplotlib.backend_bases import key_press_handler
from mpl_toolkits.mplot3d.art3d import Line3DCollection

import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from itertools import product


## Parámetros

In [51]:
# Rutas de directorios
DATA_DIR = Path("data")
OUTPUT_DIR = Path("resultados")

# Parámetros de filtrado y preprocesamiento
GROUND_Z = 0.3      # Altura máxima del suelo en metros (se excluye lo que esté por debajo)
VOXEL_SIZE = 0.15   # Tamaño del vóxel en metros para el submuestreo

# Parámetros de detección (DBSCAN)
DBSCAN_EPS = 1.8
DBSCAN_MIN_SAMPLES = 8
MIN_POINTS = 80     # Mínimo de puntos que debe tener un clúster para ser considerado vehículo

# Parámetros de Tracking (Seguimiento)
TRACK_GATE = 5.0    # Distancia máxima de asociación en metros
TRACK_MAX_GAP = 1.0 # Tiempo máximo tolerado sin detección en segundos
TRACK_MIN_HITS = 3  # Detecciones consecutivas requeridas para confirmar un ID de vehículo

## Funciones de Carga de Datos

In [52]:
def timestamp_ns(path):
    """Extrae el tiempo en nanosegundos del nombre del archivo.
    Los nanosegundos del nombre no están rellenados con ceros."""
    seconds, nanos = map(int, path.stem.split('_')[1:])
    return seconds * 1_000_000_000 + nanos


def load_points(path):
    """Carga la nube de puntos desde un archivo CSV a un array de numpy filtrando valores no finitos."""
    points = np.loadtxt(path, delimiter=',', skiprows=1, ndmin=2)
    if points.size == 0:
        return np.empty((0, 3))
    if points.shape[1] != 3:
        raise ValueError(f'{path}: se esperaban las columnas x,y,z')
    return points[np.isfinite(points).all(axis=1)]

## Detección de Vehículos (DBSCAN)

In [53]:
def detect(points):
    # En estos datos Z es la altura. Se excluye la banda próxima al suelo.
    ground = points[:, 2] <= GROUND_Z
    objects = points[~ground]
    
    if len(objects):
        # Un punto por vóxel evita que la densidad cercana domine DBSCAN.
        _, indices = np.unique(np.floor(objects / VOXEL_SIZE), axis=0,
                               return_index=True)
        objects = objects[indices]
        
    labels = (DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
              .fit_predict(objects) if len(objects) else np.empty(0, dtype=int))
              
    detections = []
    for label in sorted(set(labels) - {-1}):
        cluster = objects[labels == label]
        low, high = cluster.min(axis=0), cluster.max(axis=0)
        size = high - low
        
        # También admite vehículos parcialmente visibles en los bordes.
        if len(cluster) < MIN_POINTS or size[1] < .8 or size[2] < .6:
            continue
            
        detections.append({'center': (low[:2] + high[:2]) / 2,
                           'low': low, 'high': high, 'points': len(cluster)})
                           
    return detections, int(ground.sum())

## Seguimiento de Vehículos (Tracking)

In [54]:
@dataclass
class Track:
    id: int
    center: np.ndarray
    time: float
    velocity: np.ndarray = field(default_factory=lambda: np.zeros(2))
    hits: int = 1
    streak: int = 1
    confirmed: bool = False

    def predict(self, time):
        return self.center + self.velocity * (time - self.time)


class Tracker:
    def __init__(self, gate=TRACK_GATE, max_gap=TRACK_MAX_GAP, min_hits=TRACK_MIN_HITS):
        self.gate = gate
        self.max_gap = max_gap
        self.min_hits = min_hits
        self.active = []
        self.tracks = []

    def update(self, detections, time):
        self.active = [t for t in self.active if time - t.time <= self.max_gap]
        assigned = {}
        if self.active and detections:
            predicted = np.array([t.predict(time) for t in self.active])
            centers = np.array([d['center'] for d in detections])
            distances = np.linalg.norm(predicted[:, None] - centers[None], axis=2)
            
            # Bloquear parejas imposibles ANTES de la asignación global.
            cost = np.where(distances <= self.gate, distances, 1e9)
            rows, cols = linear_sum_assignment(cost)
            
            for row, col in zip(rows, cols):
                if distances[row, col] > self.gate:
                    continue
                track = self.active[row]
                dt = time - track.time
                if dt > 0:
                    measured = (centers[col] - track.center) / dt
                    track.velocity = .8 * track.velocity + .2 * measured
                track.center, track.time = centers[col], time
                track.hits += 1
                track.streak += 1
                track.confirmed |= track.streak >= self.min_hits
                assigned[col] = track.id
                
        matched = set(assigned.values())
        for track in self.active:
            if track.id not in matched:
                track.streak = 0
                
        for i, detection in enumerate(detections):
            if i not in assigned:
                track = Track(len(self.tracks) + 1, detection['center'], time,
                              confirmed=self.min_hits <= 1)
                self.tracks.append(track)
                self.active.append(track)
                assigned[i] = track.id
                
        return assigned

## Funciones de Visualización 3D

In [55]:
def visualize(files, frames, observations, clouds=None):
    if clouds is None:
        clouds = [load_points(path) for path in files]
        
    by_frame = {}
    for row in observations:
        by_frame.setdefault(row['frame'], []).append(row)
        
    vehicle_clouds = [vehicle_points(points, by_frame.get(i, []), GROUND_Z, VOXEL_SIZE)
                      for i, points in enumerate(clouds)]

    palette = px.colors.qualitative.Plotly

    # Calcular límites globales
    if observations:
        g_xmin = min(row['x_min'] for row in observations) - 5
        g_xmax = max(row['x_max'] for row in observations) + 5
        g_ymin = min(row['y_min'] for row in observations) - 5
        g_ymax = max(row['y_max'] for row in observations) + 5
        g_zmin = min(row['z_min'] for row in observations) - 2
        g_zmax = max(row['z_max'] for row in observations) + 5
        
        # Encontrar el máximo número de vehículos en un solo frame para pre-crear los trazos
        max_vehicles = max(len(by_frame.get(i, [])) for i in range(len(frames)))
    else:
        g_xmin, g_xmax, g_ymin, g_ymax, g_zmin, g_zmax = -50, 50, -50, 50, -5, 10
        max_vehicles = 0

    x_range = g_xmax - g_xmin
    y_range = g_ymax - g_ymin
    z_range = g_zmax - g_zmin
    max_range = max(x_range, y_range, z_range)

    fig = go.FigureWidget()
    
    fig.update_layout(
        margin=dict(l=0, r=0, b=0, t=40),
        scene=dict(
            xaxis=dict(visible=False, range=[g_xmin, g_xmax], autorange=False), 
            yaxis=dict(visible=False, range=[g_ymin, g_ymax], autorange=False), 
            zaxis=dict(visible=False, range=[g_zmin, g_zmax], autorange=False), 
            aspectmode='manual',
            aspectratio=dict(x=x_range/max_range, y=y_range/max_range, z=z_range/max_range),
            camera=dict(eye=dict(x=-1.2, y=-1.5, z=2.5)) 
        ),
        height=700,
        showlegend=False,
        uirevision='constant' 
    )

    # PRE-CREAR TRAZOS VACÍOS
    # Por cada vehículo posible, necesitamos 3 trazos: [0] puntos, [1] caja, [2] texto
    for _ in range(max_vehicles):
        # Trazos para puntos
        fig.add_trace(go.Scatter3d(x=[], y=[], z=[], mode='markers', marker=dict(size=2, opacity=0.8), showlegend=False))
        # Trazos para líneas de la caja
        fig.add_trace(go.Scatter3d(x=[], y=[], z=[], mode='lines', line=dict(width=4), showlegend=False, hoverinfo='none'))
        # Trazos para texto
        fig.add_trace(go.Scatter3d(x=[], y=[], z=[], mode='text', text=[], textfont=dict(size=16), showlegend=False))

    def update_frame(index):
        with fig.batch_update():
            fig.layout.title = f"Frame {index}/{len(frames) - 1} | Vehículos: {frames[index]['vehiculos']} | Tiempo: {frames[index]['tiempo_s']:.2f} s"
            
            vehiculos_actuales = by_frame.get(index, [])
            nubes_actuales = vehicle_clouds[index]
            
            # Actualizamos los trazos existentes
            for i in range(max_vehicles):
                # Índices de los 3 trazos correspondientes a este vehículo
                idx_points = i * 3
                idx_box = i * 3 + 1
                idx_text = i * 3 + 2
                
                if i < len(vehiculos_actuales):
                    # Hay un vehículo para este "slot"
                    row = vehiculos_actuales[i]
                    points = nubes_actuales[i]
                    color = palette[(row['id'] - 1) % len(palette)]
                    
                    # 1. Actualizar Puntos
                    if len(points) > 0:
                        fig.data[idx_points].x = points[:, 0]
                        fig.data[idx_points].y = points[:, 1]
                        fig.data[idx_points].z = points[:, 2]
                    else:
                        fig.data[idx_points].x = []
                        fig.data[idx_points].y = []
                        fig.data[idx_points].z = []
                    fig.data[idx_points].marker.color = color
                    
                    # 2. Actualizar Caja
                    min_x, max_x = row['x_min'], row['x_max']
                    min_y, max_y = row['y_min'], row['y_max']
                    min_z, max_z = row['z_min'], row['z_max']
                    
                    x_lines = [min_x, max_x, max_x, min_x, min_x, None, min_x, max_x, max_x, min_x, min_x, None, min_x, min_x, None, max_x, max_x, None, max_x, max_x, None, min_x, min_x]
                    y_lines = [min_y, min_y, max_y, max_y, min_y, None, min_y, min_y, max_y, max_y, min_y, None, min_y, min_y, None, min_y, min_y, None, max_y, max_y, None, max_y, max_y]
                    z_lines = [min_z, min_z, min_z, min_z, min_z, None, max_z, max_z, max_z, max_z, max_z, None, min_z, max_z, None, min_z, max_z, None, min_z, max_z, None, min_z, max_z]
                    
                    fig.data[idx_box].x = x_lines
                    fig.data[idx_box].y = y_lines
                    fig.data[idx_box].z = z_lines
                    fig.data[idx_box].line.color = color
                    
                    # 3. Actualizar Texto
                    fig.data[idx_text].x = [row['x']]
                    fig.data[idx_text].y = [row['y']]
                    fig.data[idx_text].z = [max_z + 0.8]
                    fig.data[idx_text].text = [f"<b>ID: {row['id']}</b>"]
                    fig.data[idx_text].textfont.color = color
                    
                else:
                    # No hay vehículo para este "slot", limpiamos los datos para ocultarlo
                    fig.data[idx_points].x = []
                    fig.data[idx_points].y = []
                    fig.data[idx_points].z = []
                    
                    fig.data[idx_box].x = []
                    fig.data[idx_box].y = []
                    fig.data[idx_box].z = []
                    
                    fig.data[idx_text].x = []
                    fig.data[idx_text].y = []
                    fig.data[idx_text].z = []
                    fig.data[idx_text].text = []

    play = widgets.Play(
        value=0,
        min=0,
        max=len(frames)-1,
        step=1,
        interval=100, 
        description="Reproducir",
        show_repeat=False
    )
    
    slider = widgets.IntSlider(min=0, max=len(frames)-1, step=1, description='Frame:')
    widgets.jslink((play, 'value'), (slider, 'value'))
    
    def on_slider_change(change):
        update_frame(change.new)
        
    slider.observe(on_slider_change, names='value')
    
    controls = widgets.HBox([play, slider])
    
    display(controls, fig)
    update_frame(0)

## Procesamiento Principal y Resultados

In [56]:
# 1. Búsqueda de archivos
files = sorted(DATA_DIR.glob('pointcloud_*.csv'), key=timestamp_ns)
if not files:
    raise ValueError(f'No se han encontrado frames en {DATA_DIR}')

# Inicialización
tracker = Tracker()
start = timestamp_ns(files[0])
frames, observations = [], []
clouds = [] 

print("Iniciando procesamiento...")

# 2. Bucle principal por cada frame
for frame, path in enumerate(files):
    time = (timestamp_ns(path) - start) / 1e9
    points = load_points(path)
    clouds.append(points)
    
    # Detección y Tracking
    detections, ground_count = detect(points)
    ids = tracker.update(detections, time)
    
    # Registro de datos del frame
    frames.append({
        'frame': frame, 
        'archivo': path.name, 
        'tiempo_s': time,
        'puntos_validos': len(points), 
        'puntos_suelo': ground_count,
        'candidatos_dbscan': len(detections)
    })
    
    # Registro de las observaciones (bounding boxes)
    for i, detection in enumerate(detections):
        low, high = detection['low'], detection['high']
        observations.append({
            'frame': frame, 'tiempo_s': time, 'id': ids[i],
            'x': detection['center'][0], 'y': detection['center'][1],
            'x_min': low[0], 'y_min': low[1], 'z_min': low[2],
            'x_max': high[0], 'y_max': high[1], 'z_max': high[2],
            'puntos': detection['points']
        })
        
    if (frame + 1) % 50 == 0:
        print(f'Procesados {frame + 1}/{len(files)} frames')

# 3. Confirmación retrospectiva
confirmed = {t.id for t in tracker.tracks if t.confirmed}
observations = [row for row in observations if row['id'] in confirmed]

for frame in frames:
    ids = sorted(row['id'] for row in observations if row['frame'] == frame['frame'])
    frame.update(vehiculos=len(ids), ids=';'.join(map(str, ids)))

# 4. Exportar resultados con Pandas
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_frames = pd.DataFrame(frames)
df_tracking = pd.DataFrame(observations)

# Reordenar columnas para tracking.csv como en el original
columnas_tracking = ['frame', 'tiempo_s', 'id', 'x', 'y', 'x_min', 'y_min', 'z_min',
                     'x_max', 'y_max', 'z_max', 'puntos']
df_tracking = df_tracking[columnas_tracking]

df_frames.to_csv(OUTPUT_DIR / 'conteo_frames.csv', index=False, encoding='utf-8')
df_tracking.to_csv(OUTPUT_DIR / 'tracking.csv', index=False, encoding='utf-8')

# Resumen
summary = (
    f'Frames: {len(files)}\n'
    f'Duración: {frames[-1]["tiempo_s"]:.3f} s\n'
    f'Vehículos únicos (IDs confirmados): {len(confirmed)}\n'
    f'Máximo de vehículos detectados en un frame: {max(f["vehiculos"] for f in frames)}\n'
    f'Suelo: z <= {GROUND_Z} m; vóxel: {VOXEL_SIZE} m\n'
    f'DBSCAN: eps={DBSCAN_EPS} m; min_samples={DBSCAN_MIN_SAMPLES}; min_points={MIN_POINTS}\n'
    f'Tracking: gate={TRACK_GATE} m; max_gap={TRACK_MAX_GAP} s; min_hits={TRACK_MIN_HITS}\n'
)
(OUTPUT_DIR / 'resumen.txt').write_text(summary, encoding='utf-8')
print("\n" + summary)

# 5. Visualización
print("Preparando visualización 3D interactiva...")
visualize(files, frames, observations, clouds)

Iniciando procesamiento...
Procesados 50/369 frames
Procesados 100/369 frames
Procesados 150/369 frames
Procesados 200/369 frames
Procesados 250/369 frames
Procesados 300/369 frames
Procesados 350/369 frames

Frames: 369
Duración: 18.652 s
Vehículos únicos (IDs confirmados): 4
Máximo de vehículos detectados en un frame: 3
Suelo: z <= 0.3 m; vóxel: 0.15 m
DBSCAN: eps=1.8 m; min_samples=8; min_points=80
Tracking: gate=5.0 m; max_gap=1.0 s; min_hits=3

Preparando visualización 3D interactiva...


FigureWidget({
    'data': [{'marker': {'opacity': 0.8, 'size': 2},
              'mode': 'markers',
              'showlegend': False,
              'type': 'scatter3d',
              'uid': '2665b8cd-33d8-44a0-9d2e-f89ad9129391',
              'x': [],
              'y': [],
              'z': []},
             {'hoverinfo': 'none',
              'line': {'width': 4},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter3d',
              'uid': '8839360d-b756-4ab3-8b88-4475ce956448',
              'x': [],
              'y': [],
              'z': []},
             {'mode': 'text',
              'showlegend': False,
              'text': [],
              'textfont': {'size': 16},
              'type': 'scatter3d',
              'uid': 'abf69cd4-e1fe-45ad-8515-2b2ec5c96087',
              'x': [],
              'y': [],
              'z': []},
             {'marker': {'opacity': 0.8, 'size': 2},
              'mode': 'markers',
            